In [ ]:
from tensorflow.keras.datasets import mnist

(x_train, y_train), (x_test, y_test) = mnist.load_data()

print(x_train.shape, y_train.shape)
print(x_test.shape, y_test.shape)


In [ ]:
# Step 1: Install required libraries (only once per Colab runtime)
!pip install tensorflow scikit-learn matplotlib pandas


In [ ]:
# Step 2: Import dependencies
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import confusion_matrix, classification_report

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models, optimizers


In [ ]:
# Step 1: Import libraries
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder

# Step 2: Load your uploaded MNIST file
data = np.load('mnist.npz')   # File you uploaded in left panel

X_train_full = data['x_train']
y_train_full = data['y_train']
X_test_full = data['x_test']
y_test_full = data['y_test']

print("Train:", X_train_full.shape, "Test:", X_test_full.shape)


In [ ]:
# Combine training and testing for unified splitting
X_full = np.concatenate([X_train_full, X_test_full], axis=0)
y_full = np.concatenate([y_train_full, y_test_full], axis=0)

# Normalize and reshape
X_full = X_full.astype('float32') / 255.0  # MNIST pixel range 0–255
X_full = np.expand_dims(X_full, -1)        # (n, 28, 28, 1)

# One-hot encode labels
encoder = OneHotEncoder(sparse_output=False)
y_full_oh = encoder.fit_transform(y_full.reshape(-1, 1))

# Split 70/15/15
X_train_val, X_test, y_train_val, y_test = train_test_split(
    X_full, y_full_oh, test_size=0.15, random_state=42, stratify=y_full
)
val_fraction = 0.15 / 0.85
X_train, X_val, y_train, y_val = train_test_split(
    X_train_val, y_train_val, test_size=val_fraction, random_state=42,
    stratify=np.argmax(y_train_val, axis=1)
)

print("Train:", X_train.shape, "Val:", X_val.shape, "Test:", X_test.shape)


In [ ]:
# === Q14: Load & configure AlexNet and GoogLeNet for fine-tuning on MNIST (PyTorch) ===
# (Colab: Runtime > Change runtime type > GPU)

import torch, numpy as np
from torch import nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.models import alexnet, AlexNet_Weights, googlenet, GoogLeNet_Weights
from PIL import Image

SEED   = 42
BATCH  = 64
IMG_SZ = 224  # Keep 224 to match your TF pipeline
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

torch.manual_seed(SEED)
np.random.seed(SEED)

# ---------- Dataset builder: grayscale->RGB, resize, official ImageNet normalize ----------
class NumpyImageDataset(Dataset):
    def __init__(self, X, y, tfm):
        self.X = X
        # one-hot -> class index if needed
        self.y = np.argmax(y, axis=1) if (y.ndim == 2 and y.shape[1] > 1) else y
        self.tfm = tfm

    def __len__(self): return len(self.X)

    def __getitem__(self, idx):
        img = self.X[idx]
        # scale to uint8 if needed
        if img.dtype != np.uint8:
            img = (img * 255.0).clip(0, 255).astype(np.uint8)
        if img.ndim == 3 and img.shape[-1] == 1:
            img = img.squeeze(-1)  # HxW
        pil = Image.fromarray(img).convert("RGB")
        return self.tfm(pil), int(self.y[idx])

# ---------- Transforms from official pretrained weights ----------
alex_w = AlexNet_Weights.IMAGENET1K_V1
gn_w   = GoogLeNet_Weights.IMAGENET1K_V1

alex_tfm = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(IMG_SZ),
    alex_w.transforms(),  # ToTensor + Normalize(mean/std)
])

gn_tfm = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(IMG_SZ),
    gn_w.transforms(),
])

# ---------- Infer NUM_CLASSES from your labels exactly like in TF ----------
NUM_CLASSES = (y_train.shape[1] if (y_train.ndim == 2 and y_train.shape[1] > 1)
               else int(np.max(y_train)) + 1)

# ---------- Build DataLoaders (separate, because transforms differ per weights) ----------
alex_train = DataLoader(NumpyImageDataset(X_train, y_train, alex_tfm), batch_size=BATCH, shuffle=True)
alex_val   = DataLoader(NumpyImageDataset(X_val,   y_val,   alex_tfm), batch_size=BATCH, shuffle=False)
alex_test  = DataLoader(NumpyImageDataset(X_test,  y_test,  alex_tfm), batch_size=BATCH, shuffle=False)

gn_train = DataLoader(NumpyImageDataset(X_train, y_train, gn_tfm), batch_size=BATCH, shuffle=True)
gn_val   = DataLoader(NumpyImageDataset(X_val,   y_val,   gn_tfm), batch_size=BATCH, shuffle=False)
gn_test  = DataLoader(NumpyImageDataset(X_test,  y_test,  gn_tfm), batch_size=BATCH, shuffle=False)

# ---------- Model builders: load true ImageNet-pretrained, replace classifier heads ----------
def build_alexnet(num_classes):
    m = alexnet(weights=alex_w)
    m.classifier[6] = nn.Linear(m.classifier[6].in_features, num_classes)
    return m

def build_googlenet(num_classes):
    m = googlenet(weights=gn_w, aux_logits=True)  # aux heads used during training
    m.fc = nn.Linear(m.fc.in_features, num_classes)
    if m.aux_logits:
        m.aux1.fc = nn.Linear(m.aux1.fc.in_features, num_classes)
        m.aux2.fc = nn.Linear(m.aux2.fc.in_features, num_classes)
    return m

alex_model = build_alexnet(NUM_CLASSES).to(DEVICE)
gn_model   = build_googlenet(NUM_CLASSES).to(DEVICE)

# ---------- Stage-1 setup (feature extractor; backbone frozen) ----------
def freeze_backbone_stage1(model_name, model):
    # Freeze everything first
    for p in model.parameters():
        p.requires_grad = False
    # Unfreeze only final classifier layers (like your TF compile_head_training)
    if model_name == "alexnet":
        for p in model.classifier.parameters():
            p.requires_grad = True
    elif model_name == "googlenet":
        for p in model.fc.parameters():
            p.requires_grad = True
        if getattr(model, "aux_logits", False):
            for p in model.aux1.fc.parameters(): p.requires_grad = True
            for p in model.aux2.fc.parameters(): p.requires_grad = True

freeze_backbone_stage1("alexnet",   alex_model)
freeze_backbone_stage1("googlenet", gn_model)

# ---------- Optional sanity checks (forward pass, no training) ----------
alex_model.eval()
xb, yb = next(iter(alex_train))
with torch.no_grad():
    out = alex_model(xb.to(DEVICE))
    if hasattr(out, "logits"):  # just in case (not expected for AlexNet)
        out = out.logits
print("AlexNet forward pass OK:", tuple(out.shape))

gn_model.eval()
xb, yb = next(iter(gn_train))
with torch.no_grad():
    out = gn_model(xb.to(DEVICE))
    # GoogLeNet may return a named output; normalize to main logits for shape print
    if hasattr(out, "logits"):
        out = out.logits
    elif isinstance(out, tuple):
        out = out[0]
print("GoogLeNet forward pass OK:", tuple(out.shape))

# ---------- Prepare for Stage-2 (fine-tuning; unfreeze top layers) ----------
def prepare_for_finetune(model_name, model):
    # Default: unfreeze entire network (safe & simple, like setting all trainable True)
    for p in model.parameters():
        p.requires_grad = True
    # If you want to mimic "unfreeze only top blocks", you can keep early parts frozen:
    if model_name == "googlenet":
        # Example: keep early inception blocks frozen; fine-tune last ones + fc
        for p in model.inception3a.parameters(): p.requires_grad = False
        for p in model.inception3b.parameters(): p.requires_grad = False
        for p in model.inception4a.parameters(): p.requires_grad = False
        for p in model.inception4b.parameters(): p.requires_grad = False
        for p in model.inception4c.parameters(): p.requires_grad = False
        # Leave 4d, 4e, 5a, 5b, fc trainable (you can tweak as desired)

# Call this now (like your TF `prepare_for_finetune(...)`), but we won't start training yet.
prepare_for_finetune("alexnet",   alex_model)
prepare_for_finetune("googlenet", gn_model)

# (Optional) brief summaries: count trainable params (similar to model.summary in TF)
alex_trainable = sum(p.numel() for p in alex_model.parameters() if p.requires_grad)
gn_trainable   = sum(p.numel() for p in gn_model.parameters()   if p.requires_grad)
print(f"AlexNet trainable params (Stage-2 ready): {alex_trainable}")
print(f"GoogLeNet trainable params (Stage-2 ready): {gn_trainable}")

# ===== Q15 (next cell): we'll create optimizers, loss, and call the training loop =====


In [ ]:
# ===== Q15: Train the fine-tuned models on the SAME splits as Part-1 (PyTorch) =====
# Assumes from Q14 you already have:
#   alex_model, gn_model
#   alex_train, alex_val
#   gn_train,   gn_val
# and the original arrays: y_train, y_val (for a quick sanity check below)

import csv, os, torch
from torch import nn, optim

EPOCHS_TOTAL = 20
EPOCHS_HEAD  = 5   # Stage-1: train head only
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ---- sanity: label dimensions match (one-hot or class indices) ----
def _num_classes_from_y(y):
    return y.shape[1] if (y.ndim == 2 and y.shape[1] > 1) else int(y.max()) + 1

assert _num_classes_from_y(y_train) == _num_classes_from_y(y_val), \
    "Label dimensions mismatch; check splits are the SAME as Part-1."

# ---- training/eval helpers (handles GoogLeNet aux heads) ----
def train_one_epoch(model, loader, optimizer, criterion, aux_coeff=0.3):
    model.train()
    total_loss, correct, n = 0.0, 0, 0
    for images, labels in loader:
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        outputs = model(images)

        # Normalize output forms (GoogLeNet may return named/tuple with aux)
        if hasattr(outputs, "logits"):             # named output
            main_logits = outputs.logits
            loss = criterion(main_logits, labels)
            if getattr(model, "aux_logits", False):
                if hasattr(outputs, "aux_logits1"):
                    loss = loss + aux_coeff * criterion(outputs.aux_logits1, labels)
                if hasattr(outputs, "aux_logits2"):
                    loss = loss + aux_coeff * criterion(outputs.aux_logits2, labels)
        elif isinstance(outputs, tuple):           # tuple: (main, aux2, aux1)
            main_logits, aux2, aux1 = outputs
            loss = criterion(main_logits, labels)
            if getattr(model, "aux_logits", False):
                if aux1 is not None: loss = loss + aux_coeff * criterion(aux1, labels)
                if aux2 is not None: loss = loss + aux_coeff * criterion(aux2, labels)
        else:
            main_logits = outputs
            loss = criterion(main_logits, labels)

        loss.backward()
        optimizer.step()

        total_loss += loss.item() * images.size(0)
        correct += (main_logits.argmax(1) == labels).sum().item()
        n += images.size(0)
    return total_loss / n, correct / n

@torch.no_grad()
def evaluate(model, loader, criterion):
    model.eval()
    total_loss, correct, n = 0.0, 0, 0
    for images, labels in loader:
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        outputs = model(images)
        if hasattr(outputs, "logits"):
            logits = outputs.logits
        elif isinstance(outputs, tuple):
            logits = outputs[0]
        else:
            logits = outputs
        loss = criterion(logits, labels)
        total_loss += loss.item() * images.size(0)
        correct += (logits.argmax(1) == labels).sum().item()
        n += images.size(0)
    return total_loss / n, correct / n

# ---- stage-freeze helpers reused from Q14 (redefine here if needed) ----
def freeze_backbone_stage1(model_name, model):
    for p in model.parameters():
        p.requires_grad = False
    if model_name == "alexnet":
        for p in model.classifier.parameters():
            p.requires_grad = True
    elif model_name == "googlenet":
        for p in model.fc.parameters():
            p.requires_grad = True
        if getattr(model, "aux_logits", False):
            for p in model.aux1.fc.parameters(): p.requires_grad = True
            for p in model.aux2.fc.parameters(): p.requires_grad = True

def prepare_for_finetune(model_name, model):
    # simplest: unfreeze everything for low-LR fine-tuning
    for p in model.parameters():
        p.requires_grad = True
    # (Optional) keep early GoogLeNet blocks frozen; fine-tune upper ones:
    if model_name == "googlenet":
        for p in model.inception3a.parameters(): p.requires_grad = False
        for p in model.inception3b.parameters(): p.requires_grad = False
        for p in model.inception4a.parameters(): p.requires_grad = False
        for p in model.inception4b.parameters(): p.requires_grad = False
        for p in model.inception4c.parameters(): p.requires_grad = False

# ---- CSV logger helper (append across stages) ----
def _init_csv(path):
    with open(path, "w", newline="") as f:
        w = csv.writer(f)
        w.writerow(["epoch", "train_loss", "train_acc", "val_loss", "val_acc"])

def _append_csv(path, epoch_idx, tr_loss, tr_acc, va_loss, va_acc):
    with open(path, "a", newline="") as f:
        w = csv.writer(f)
        w.writerow([epoch_idx, f"{tr_loss:.6f}", f"{tr_acc:.6f}", f"{va_loss:.6f}", f"{va_acc:.6f}"])

# ---- two-stage trainer (mirrors your Keras function) ----
def train_two_stages(model_name, model, train_loader, val_loader, arch_name):
    """
    Stage-1: freeze backbone, train head for 5 epochs (lr=1e-3)
    Stage-2: unfreeze for fine-tuning until epoch 20 (lr=1e-4)
    Logs to {arch_name}_log.csv and saves best checkpoints.
    """
    criterion = nn.CrossEntropyLoss()

    log_path = f"{arch_name}_log.csv"
    _init_csv(log_path)

    # -------- Stage 1: feature extractor (freeze backbone) --------
    freeze_backbone_stage1(model_name, model)
    opt1 = optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-3)
    best_val_acc_s1, best_path_s1 = 0.0, f"{arch_name}_best_head.pth"

    for epoch in range(1, EPOCHS_HEAD + 1):
        tr_loss, tr_acc = train_one_epoch(model, train_loader, opt1, criterion)
        va_loss, va_acc = evaluate(model, val_loader, criterion)
        _append_csv(log_path, epoch, tr_loss, tr_acc, va_loss, va_acc)
        print(f"[{arch_name}][S1][{epoch}/{EPOCHS_HEAD}] "
              f"train_loss={tr_loss:.4f} acc={tr_acc:.4f} | val_loss={va_loss:.4f} acc={va_acc:.4f}")
        if va_acc > best_val_acc_s1:
            best_val_acc_s1 = va_acc
            torch.save(model.state_dict(), best_path_s1)

    # -------- Stage 2: fine-tune (unfreeze, small LR) --------
    prepare_for_finetune(model_name, model)
    opt2 = optim.AdamW(model.parameters(), lr=1e-4)
    best_val_acc_s2, best_path_s2 = 0.0, f"{arch_name}_best_finetuned.pth"

    for epoch in range(EPOCHS_HEAD + 1, EPOCHS_TOTAL + 1):
        tr_loss, tr_acc = train_one_epoch(model, train_loader, opt2, criterion)
        va_loss, va_acc = evaluate(model, val_loader, criterion)
        _append_csv(log_path, epoch, tr_loss, tr_acc, va_loss, va_acc)
        print(f"[{arch_name}][S2][{epoch}/{EPOCHS_TOTAL}] "
              f"train_loss={tr_loss:.4f} acc={tr_acc:.4f} | val_loss={va_loss:.4f} acc={va_acc:.4f}")
        if va_acc > best_val_acc_s2:
            best_val_acc_s2 = va_acc
            torch.save(model.state_dict(), best_path_s2)

    return {"best_head": best_val_acc_s1, "best_ft": best_val_acc_s2}

# ===== Actually train both models (SAME train/val splits as Part-1) =====
alex_stats = train_two_stages(
    model_name="alexnet", model=alex_model,
    train_loader=alex_train, val_loader=alex_val, arch_name="alexnet"
)

gn_stats = train_two_stages(
    model_name="googlenet", model=gn_model,
    train_loader=gn_train, val_loader=gn_val, arch_name="googlenet"
)

print("Q15 training complete. Logs written to alexnet_log.csv and googlenet_log.csv")
print("Best Val Acc — AlexNet:", alex_stats)
print("Best Val Acc — GoogLeNet:", gn_stats)


In [ ]:
# ===== Q16: Record & export per-epoch losses for AlexNet and GoogLeNet (PyTorch logs) =====
import pandas as pd
import matplotlib.pyplot as plt
import os

# 1) Load the logs created in Q15
assert os.path.exists("alexnet_log.csv"), "alexnet_log.csv not found. Re-run Q15 training."
assert os.path.exists("googlenet_log.csv"), "googlenet_log.csv not found. Re-run Q15 training."

alex = pd.read_csv("alexnet_log.csv")      # columns: epoch, train_loss, train_acc, val_loss, val_acc
gnet = pd.read_csv("googlenet_log.csv")

# 2) Keep epoch + losses; rename train_loss -> loss for parity with your TF code
def select_and_standardize(df):
    cols = []
    if "epoch" in df.columns: cols.append("epoch")
    if "train_loss" in df.columns: cols.append("train_loss")
    if "val_loss" in df.columns: cols.append("val_loss")
    out = df[cols].copy()
    if "train_loss" in out.columns:
        out = out.rename(columns={"train_loss": "loss"})
    # Make epochs 1-indexed if they start at 0
    if "epoch" in out.columns and out["epoch"].min() == 0:
        out["epoch"] = out["epoch"] + 1
    return out

alex = select_and_standardize(alex)
gnet = select_and_standardize(gnet)

alex["model"] = "AlexNet"
gnet["model"] = "GoogLeNet"

# 3) Save clean per-epoch CSVs for your report archive
alex.to_csv("q16_epoch_losses_alexnet.csv", index=False)
gnet.to_csv("q16_epoch_losses_googlenet.csv", index=False)

# Also a combined file for easier plotting/inspection
combined = pd.concat([alex, gnet], ignore_index=True)
combined.to_csv("q16_epoch_losses_combined.csv", index=False)

# 4) Plot training vs validation loss for each model (separate figures and an overlaid one)
def plot_losses(df, title, fname):
    plt.figure(figsize=(7,5))
    plt.plot(df["epoch"], df["loss"], label="train loss")
    plt.plot(df["epoch"], df["val_loss"], "--", label="val loss")
    plt.xlabel("Epoch"); plt.ylabel("Loss"); plt.title(title)
    plt.grid(True, alpha=0.3); plt.legend()
    plt.tight_layout(); plt.savefig(fname, dpi=150); plt.show()

plot_losses(alex, "AlexNet: Training vs Validation Loss", "q16_alexnet_loss.png")
plot_losses(gnet, "GoogLeNet: Training vs Validation Loss", "q16_googlenet_loss.png")

# Overlaid plot (validation only)
plt.figure(figsize=(8,5))
plt.plot(alex["epoch"], alex["val_loss"], label="AlexNet val loss")
plt.plot(gnet["epoch"], gnet["val_loss"], label="GoogLeNet val loss")
plt.xlabel("Epoch"); plt.ylabel("Loss"); plt.title("Validation Loss (AlexNet vs GoogLeNet)")
plt.grid(True, alpha=0.3); plt.legend()
plt.tight_layout(); plt.savefig("q16_val_loss_overlay.png", dpi=150); plt.show()

print("Saved: q16_epoch_losses_alexnet.csv, q16_epoch_losses_googlenet.csv, q16_epoch_losses_combined.csv")
print("Saved figures: q16_alexnet_loss.png, q16_googlenet_loss.png, q16_val_loss_overlay.png")


In [ ]:
# ===== Q17: Evaluate on test set and record metrics (PyTorch) =====
import numpy as np
import pandas as pd
import torch
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    confusion_matrix, classification_report
)

# Assumes you still have:
#   alex_model, gn_model   (trained in Q15)
#   alex_test,  gn_test    (DataLoaders built in Q14)
# And NUM_CLASSES from earlier:
try:
    NUM_CLASSES
except NameError:
    NUM_CLASSES = 10
CLASS_NAMES = [str(i) for i in range(NUM_CLASSES)]

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

@torch.no_grad()
def evaluate_and_save(model, test_loader, arch_name, class_names=None):
    """Evaluate a trained PyTorch model on a test DataLoader and save metrics/artifacts."""
    model.eval()
    y_true, y_pred = [], []

    for images, labels in test_loader:
        images = images.to(DEVICE)
        labels = labels.to(DEVICE)

        outputs = model(images)
        # Handle GoogLeNet's output variants; use main logits only
        if hasattr(outputs, "logits"):
            logits = outputs.logits
        elif isinstance(outputs, tuple):
            logits = outputs[0]
        else:
            logits = outputs

        preds = logits.argmax(dim=1)

        y_pred.extend(preds.cpu().numpy().tolist())
        y_true.extend(labels.cpu().numpy().tolist())

    y_true = np.array(y_true, dtype=int)
    y_pred = np.array(y_pred, dtype=int)

    # --- Metrics ---
    acc        = accuracy_score(y_true, y_pred)
    prec_macro = precision_score(y_true, y_pred, average="macro", zero_division=0)
    rec_macro  = recall_score(y_true, y_pred, average="macro",  zero_division=0)
    prec_micro = precision_score(y_true, y_pred, average="micro", zero_division=0)
    rec_micro  = recall_score(y_true, y_pred, average="micro",  zero_division=0)

    # Confusion matrix (counts) + normalized
    cm = confusion_matrix(y_true, y_pred)
    cm_df = pd.DataFrame(cm,
                         index=class_names if class_names else None,
                         columns=class_names if class_names else None)
    cm_df.to_csv(f"{arch_name}_confusion_matrix.csv", index=class_names is not None)

    row_sums = cm.sum(axis=1, keepdims=True)
    row_sums[row_sums == 0] = 1  # avoid divide-by-zero
    cm_norm = cm.astype(float) / row_sums
    cm_norm_df = pd.DataFrame(cm_norm,
                              index=class_names if class_names else None,
                              columns=class_names if class_names else None)
    cm_norm_df.to_csv(f"{arch_name}_confusion_matrix_normalized.csv", index=class_names is not None)

    # Full per-class report
    report = classification_report(y_true, y_pred, output_dict=True, zero_division=0)
    pd.DataFrame(report).transpose().to_csv(f"{arch_name}_classification_report.csv")

    # Summary metrics text
    with open(f"{arch_name}_metrics.txt", "w") as f:
        f.write(
            f"test_accuracy={acc:.6f}\n"
            f"precision_macro={prec_macro:.6f}\n"
            f"recall_macro={rec_macro:.6f}\n"
            f"precision_micro={prec_micro:.6f}\n"
            f"recall_micro={rec_micro:.6f}\n"
        )

    print(f"[{arch_name}] acc={acc:.4f} | P_macro={prec_macro:.4f} | R_macro={rec_macro:.4f} "
          f"| P_micro={prec_micro:.4f} | R_micro={rec_micro:.4f}")

    return {
        "Model": arch_name,
        "Accuracy": acc,
        "Precision_macro": prec_macro,
        "Recall_macro": rec_macro,
        "Precision_micro": prec_micro,
        "Recall_micro": rec_micro
    }

# ---- Evaluate both fine-tuned models on the SAME test split ----
alex_metrics = evaluate_and_save(alex_model, alex_test, "alexnet", class_names=CLASS_NAMES)
gn_metrics   = evaluate_and_save(gn_model,   gn_test,   "googlenet", class_names=CLASS_NAMES)

# Combined summary table (handy for Q18)
summary_q17 = pd.DataFrame([alex_metrics, gn_metrics])
summary_q17.to_csv("q17_test_metrics_summary.csv", index=False)
print("\nSaved: alexnet_* and googlenet_* metric files + q17_test_metrics_summary.csv")
summary_q17


In [ ]:
# ===== Q16/Q17-style visualization: Classification report + confusion matrices (PyTorch) =====
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix
import torch

# Assumes these exist from Q14–Q15:
#   alex_model, gn_model
#   alex_test,  gn_test
# Optionally reuse DEVICE; define if missing
if "DEVICE" not in globals():
    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

NUM_CLASSES = 10
CLASS_NAMES = [str(i) for i in range(NUM_CLASSES)]

@torch.no_grad()
def report_and_plot_pt(model, test_loader, arch_name, class_names=None):
    """Print text classification report and save/show confusion-matrix heatmaps (PyTorch)."""
    model.eval()
    y_true, y_pred = [], []

    for xb, yb in test_loader:
        xb = xb.to(DEVICE)
        yb = yb.to(DEVICE)

        outputs = model(xb)
        # Handle GoogLeNet outputs
        if hasattr(outputs, "logits"):
            logits = outputs.logits
        elif isinstance(outputs, tuple):
            logits = outputs[0]
        else:
            logits = outputs

        preds = logits.argmax(dim=1)

        y_pred.extend(preds.cpu().numpy().tolist())
        y_true.extend(yb.cpu().numpy().tolist())

    y_true = np.array(y_true, dtype=int)
    y_pred = np.array(y_pred, dtype=int)

    # ---- Text report ----
    print(f"\n=== {arch_name.upper()} — Classification Report ===")
    try:
        print(classification_report(
            y_true, y_pred,
            target_names=class_names if class_names and len(class_names) == len(np.unique(y_true)) else None,
            digits=4, zero_division=0
        ))
    except ValueError:
        # Fallback if target_names length mismatches
        print(classification_report(y_true, y_pred, digits=4, zero_division=0))

    # ---- Confusion matrix: raw counts ----
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=class_names if class_names and len(class_names) == cm.shape[1] else np.arange(cm.shape[1]),
                yticklabels=class_names if class_names and len(class_names) == cm.shape[0] else np.arange(cm.shape[0]))
    plt.title(f'Confusion Matrix — {arch_name}')
    plt.xlabel('Predicted'); plt.ylabel('True')
    plt.tight_layout()
    plt.savefig(f'{arch_name}_cm_counts.png', dpi=150)
    plt.show()

    # ---- Confusion matrix: normalized by true class ----
    row_sums = cm.sum(axis=1, keepdims=True)
    row_sums[row_sums == 0] = 1
    cm_norm = cm.astype(float) / row_sums
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='Blues',
                xticklabels=class_names if class_names and len(class_names) == cm.shape[1] else np.arange(cm.shape[1]),
                yticklabels=class_names if class_names and len(class_names) == cm.shape[0] else np.arange(cm.shape[0]))
    plt.title(f'Normalized Confusion Matrix — {arch_name}')
    plt.xlabel('Predicted'); plt.ylabel('True')
    plt.tight_layout()
    plt.savefig(f'{arch_name}_cm_normalized.png', dpi=150)
    plt.show()

    # Save raw numbers (good for appendix)
    pd.DataFrame(cm,
                 index=class_names if class_names and len(class_names) == cm.shape[0] else None,
                 columns=class_names if class_names and len(class_names) == cm.shape[1] else None
                 ).to_csv(f"{arch_name}_confusion_matrix.csv",
                          index=class_names is not None and len(class_names) == cm.shape[0])

    pd.DataFrame(cm_norm,
                 index=class_names if class_names and len(class_names) == cm.shape[0] else None,
                 columns=class_names if class_names and len(class_names) == cm.shape[1] else None
                 ).to_csv(f"{arch_name}_confusion_matrix_normalized.csv",
                          index=class_names is not None and len(class_names) == cm.shape[0])

# Run for both fine-tuned models
report_and_plot_pt(alex_model, alex_test, "alexnet", CLASS_NAMES)
report_and_plot_pt(gn_model,   gn_test,   "googlenet", CLASS_NAMES)
